
# <font color="green">Pointers (square norm)</font>

## Problem

* Write a function `l2_norm_long` in assembly that computes the square norm of a three-element vector of `long`s.
* That is, translate the following C function into assembly:
```
long l2_norm_long(long * x) {
    return x[0] * x[0] + x[1] * x[1] + x[2] * x[2];
}
```
* Fill in the skeleton `l2_norm_long.s` (after `// ------- write your answer here -------`).
* The checker `check_l2_norm_long.c` verifies your result. If you see `OK`s and no errors, you are done.
* Here the pointer `x` is simply given to you. In the next two problems (*Local arrays on the stack* and *Heap allocation*) you will see where such a pointer comes from --- the stack and the heap --- and pass it to this very function.

## Hints

* Motto: "pointers are just integers and nothing more than that".
* **Pointer dereferencing.** The *Observe* cells below contain `long_ptr_deref` (which simply returns `*p`). What is generated for `*p` is
```
        ldr     x0, [x0]
```
a _load instruction_ that reads the eight bytes at the address in `x0` and puts the value in `x0`. So: a pointer parameter `p` is passed in `x0` like an integer; a pointer value is in fact an address (an integer at the assembly level); and dereferencing `p` uses a load.
* **Accessing an array element = pointer dereferencing.** The *Observe* cells also contain `array_index_long` (returning `p[0] + p[10]`). What is generated for `p[0] + p[10]` is
```
        ldr     x1, [x0]
        ldr     x0, [x0, 80]
        add     x0, x1, x0
```
The `+ 80` offset is because a single `long` takes eight bytes (10 × 8 = 80). Note that `*p` and `p[0]` end up using the same instruction --- this is where "arrays are pointers" comes from. For pointers to `int` (4 bytes), the destination registers become `w` registers and the offset for `p[10]` becomes 40.
* This problem just needs the array-indexing pattern: load `x[0]`, `x[1]`, `x[2]` (offsets 0, 8, 16), multiply each by itself, and add.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=l2_norm_long.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* Pointers are just integers (addresses); dereferencing uses load instructions. */
long long_ptr_deref(long * p) { return *p; }       /* *p           -> ldr */
long array_index_long(long * p) { return p[0] + p[10]; }  /* p[i]   -> ldr with offset */

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `l2_norm_long.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ l2_norm_long.s
	.arch armv8-a
	.file	"l2_norm_long.c"
	.text
	.align	2
	.p2align 4,,11
	.global	l2_norm_long
	.type	l2_norm_long, %function
l2_norm_long:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	l2_norm_long, .-l2_norm_long
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `l2_norm_long` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_l2_norm_long.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
long l2_norm_long(long *);

int main(int argc, char ** argv) {
  assert(argc == 4);
  long x[3] = { atol(argv[1]), atol(argv[2]), atol(argv[3]) };
  long l2 = l2_norm_long(x);
  long l2c = x[0] * x[0] + x[1] * x[1] + x[2] * x[2];
  if (l2 == l2c) {
    printf("OK %ld %ld\n", l2, l2c);
    return 0;
  } else {
    printf("NG %ld %ld\n", l2, l2c);
    return 1;
  }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `l2_norm_long.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_l2_norm_long -O3 check_l2_norm_long.c l2_norm_long.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_l2_norm_long 1 2 3
./check_l2_norm_long 3 4 5


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_l2_norm_long -O0 -g check_l2_norm_long.c l2_norm_long.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_l2_norm_long
(gdb) break l2_norm_long
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=l2_norm_long.md answer_file=l2_norm_long.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.